In [ ]:
import pickle
from pathlib import Path

import Levenshtein
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import t

## Settings

In [ ]:
# Pandas display options
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
DATASET_NAME_ORDER = [
    "iris",
    "BH_1",
    "DA",
    "alkox",
    "oer_plate_a",
    "p3ht",
    "photo_pce10",
    "photo_wf3",
    "suzuki_edbo",
    "suzuki",
]
MODEL_ORDER = [
    "gpt-5-mini-2025-08-07",
    "o4-mini-2025-04-16",
    "gpt-4.1-mini-2025-04-14",
    "gpt-4o-mini-2024-07-18",
    "claude-sonnet-4-5-20250929",
    "claude-haiku-4-5-20251001",
    "claude-3-5-haiku-20241022",
]
MODEL_LABELS = {
    "gpt-5-mini-2025-08-07": "GPT-5 mini",
    "o4-mini-2025-04-16": "o4 mini",
    "gpt-4.1-mini-2025-04-14": "GPT-4.1 mini",
    "gpt-4o-mini-2024-07-18": "GPT-4o mini",
    "claude-sonnet-4-5-20250929": "Claude Sonnet 4.5",
    "claude-haiku-4-5-20251001": "Claude Haiku 4.5",
    "claude-3-5-haiku-20241022": "Claude 3.5 Haiku",
}
COMPLETION_MODE_LIST = ["header", "header2", "row", "row2"]
COMPLETION_MODE_LABELS = {
    "header": "Header Test 1",
    "header2": "Header Test 2",
    "row": "Row Completion Test 1",
    "row2": "Row Completion Test 2",
}

In [ ]:
TIME_TAGS = [
    "20260111234227",
    "20260112114834",
    "20260112122336",
    "20260112122405",
    "20260112122431",
    "20260112122535",
]

In [ ]:
# Read and concatenate batch output logs
batch_output_logs_df_list = []
for time_tag in TIME_TAGS:
    batch_output_logs_df_tmp = pd.read_csv(
        Path("./batch_output") / f"batch_output_{time_tag}" / "batch_output_logs.csv"
    )
    batch_output_logs_df_tmp.insert(2, "time_tag", time_tag)
    batch_output_logs_df_list.append(batch_output_logs_df_tmp)
batch_output_logs_df = pd.concat(batch_output_logs_df_list, axis=0, ignore_index=True)

# Sort by dataset_name and model
batch_output_logs_df["completion_mode"] = pd.Categorical(
    batch_output_logs_df["completion_mode"],
    categories=COMPLETION_MODE_LIST,
    ordered=True,
)
batch_output_logs_df["dataset_name"] = pd.Categorical(
    batch_output_logs_df["dataset_name"], categories=DATASET_NAME_ORDER, ordered=True
)
batch_output_logs_df["model"] = pd.Categorical(
    batch_output_logs_df["model"], categories=MODEL_ORDER, ordered=True
)
batch_output_logs_df = batch_output_logs_df.sort_values(
    ["dataset_name", "model"]
).reset_index(drop=True)

# Convert to string type
batch_output_logs_df["completion_mode"] = batch_output_logs_df[
    "completion_mode"
].astype(str)
batch_output_logs_df["dataset_name"] = batch_output_logs_df["dataset_name"].astype(str)
batch_output_logs_df["model"] = batch_output_logs_df["model"].astype(str)

# Create a new column containing model parameters
batch_output_logs_df["model_w_params"] = batch_output_logs_df["model"].map(MODEL_LABELS)
for idx, row in batch_output_logs_df.iterrows():
    if pd.notna(batch_output_logs_df.loc[idx, "reasoning_effort"]):
        batch_output_logs_df.loc[idx, "model_w_params"] += (
            " " + batch_output_logs_df.loc[idx, "reasoning_effort"]
        )
    elif pd.notna(batch_output_logs_df.loc[idx, "temperature"]):
        batch_output_logs_df.loc[idx, "model_w_params"] += " temp" + str(
            round(batch_output_logs_df.loc[idx, "temperature"], 2)
        )

## Scores

In [ ]:
result_data = {}
result_scores = []

for i, batch_output_logs_df_row in batch_output_logs_df[
    ["dataset_name", "model", "model_w_params", "completion_mode", "time_tag"]
].iterrows():
    dataset_name = batch_output_logs_df_row.dataset_name
    model = batch_output_logs_df_row.model
    model_w_params = batch_output_logs_df_row.model_w_params
    completion_mode = batch_output_logs_df_row.completion_mode
    time_tag = batch_output_logs_df_row.time_tag

    print(
        f"dataset: {dataset_name}, model {model_w_params}, completion_mode {completion_mode}"
    )

    # Read original data
    dataset_orig = pd.read_csv(f"./dataset/dataset_{dataset_name}.csv")

    # Read settings
    with open(
        f"./batch_input/batch_input_{time_tag}/batch_input_settings.pkl", "rb"
    ) as f:
        settings = pickle.load(f)

    # Read results
    with open(f"./results/results_{time_tag}/results.pkl", "rb") as f:
        results = pickle.load(f)

    ids = list(settings[(dataset_name, model, completion_mode)].keys())

    matching_length_ratio_list = []
    normalized_levenshtein_distance_list = []

    for id in ids:
        test_suffix = settings[(dataset_name, model, completion_mode)][id][
            "test_suffix"
        ]
        llm_output = results[(dataset_name, model, completion_mode)][id]

        if completion_mode == "header" or completion_mode == "header2":
            length_limit = np.min([len(test_suffix), len(llm_output)])
            test_suffix = test_suffix[:length_limit]
            llm_output = llm_output[:length_limit]

        # Scores
        matching_length = 0
        for s, r in zip(test_suffix, llm_output):
            if s == r:
                matching_length += 1
            else:
                break
        matching_length_ratio = matching_length / np.max(
            [len(test_suffix), len(llm_output)]
        )
        levenshtein_distance = Levenshtein.distance(test_suffix, llm_output)
        normalized_levenshtein_distance = levenshtein_distance / np.max(
            [len(test_suffix), len(llm_output)]
        )

        matching_length_ratio_list.append(matching_length_ratio)
        normalized_levenshtein_distance_list.append(normalized_levenshtein_distance)

        if completion_mode == "row":
            test_suffix_split = test_suffix.split(",")
            llm_output_split = llm_output.split(",")

            if len(test_suffix_split) == len(llm_output_split):
                test_suffix_wo_last = ",".join(test_suffix_split[:-1])
                llm_output_wo_last = ",".join(llm_output_split[:-1])

                matching_length_wo_last = 0
                for s, r in zip(test_suffix_wo_last, llm_output_wo_last):
                    if s == r:
                        matching_length_wo_last += 1
                    else:
                        break
                matching_length_ratio_wo_last = matching_length_wo_last / np.max(
                    [len(test_suffix_wo_last), len(llm_output_wo_last)]
                )
                levenshtein_distance_wo_last = Levenshtein.distance(
                    test_suffix_wo_last, llm_output_wo_last
                )
                normalized_levenshtein_distance_wo_last = (
                    levenshtein_distance_wo_last
                    / np.max([len(test_suffix_wo_last), len(llm_output_wo_last)])
                )

                test_suffix_last = test_suffix_split[-1]
                llm_output_last = llm_output_split[-1]

                matching_length_last = 0
                for s, r in zip(test_suffix_last, llm_output_last):
                    if s == r:
                        matching_length_last += 1
                    else:
                        break
                matching_length_ratio_last = matching_length_last / np.max(
                    [len(test_suffix_last), len(llm_output_last)]
                )
                levenshtein_distance_last = Levenshtein.distance(
                    test_suffix_last, llm_output_last
                )
                normalized_levenshtein_distance_last = (
                    levenshtein_distance_last
                    / np.max([len(test_suffix_last), len(llm_output_last)])
                )

    # Scores
    matching_length_ratio_arr = np.array(matching_length_ratio_list)
    normalized_levenshtein_distance_arr = np.array(normalized_levenshtein_distance_list)

    id_matching_length_ratio_min = int(np.argmin(matching_length_ratio_arr))
    id_matching_length_ratio_max = int(np.argmax(matching_length_ratio_arr))
    id_normalized_levenshtein_distance_min = int(
        np.argmin(normalized_levenshtein_distance_arr)
    )
    id_normalized_levenshtein_distance_max = int(
        np.argmax(normalized_levenshtein_distance_arr)
    )

    if completion_mode in ["header", "header2"]:
        id_nums = [2, 4, 6, 8]
    elif completion_mode in ["row", "row2"]:
        id_nums = list(range(1, 26, 1))
    else:
        raise ValueError(f"Unknown completion_mode: {completion_mode}")

    test_prefix_matching_length_ratio_min = settings[
        (dataset_name, model, completion_mode)
    ][f"ID_{id_nums[id_matching_length_ratio_min]}"]["test_prefix"]
    test_suffix_matching_length_ratio_min = settings[
        (dataset_name, model, completion_mode)
    ][f"ID_{id_nums[id_matching_length_ratio_min]}"]["test_suffix"]
    llm_output_matching_length_ratio_min = results[
        (dataset_name, model, completion_mode)
    ][f"ID_{id_nums[id_matching_length_ratio_min]}"]
    test_prefix_matching_length_ratio_max = settings[
        (dataset_name, model, completion_mode)
    ][f"ID_{id_nums[id_matching_length_ratio_max]}"]["test_prefix"]
    test_suffix_matching_length_ratio_max = settings[
        (dataset_name, model, completion_mode)
    ][f"ID_{id_nums[id_matching_length_ratio_max]}"]["test_suffix"]
    llm_output_matching_length_ratio_max = results[
        (dataset_name, model, completion_mode)
    ][f"ID_{id_nums[id_matching_length_ratio_max]}"]
    test_prefix_normalized_levenshtein_distance_min = settings[
        (dataset_name, model, completion_mode)
    ][f"ID_{id_nums[id_normalized_levenshtein_distance_min]}"]["test_prefix"]
    test_suffix_normalized_levenshtein_distance_min = settings[
        (dataset_name, model, completion_mode)
    ][f"ID_{id_nums[id_normalized_levenshtein_distance_min]}"]["test_suffix"]
    llm_output_normalized_levenshtein_distance_min = results[
        (dataset_name, model, completion_mode)
    ][f"ID_{id_nums[id_normalized_levenshtein_distance_min]}"]
    test_prefix_normalized_levenshtein_distance_max = settings[
        (dataset_name, model, completion_mode)
    ][f"ID_{id_nums[id_normalized_levenshtein_distance_max]}"]["test_prefix"]
    test_suffix_normalized_levenshtein_distance_max = settings[
        (dataset_name, model, completion_mode)
    ][f"ID_{id_nums[id_normalized_levenshtein_distance_max]}"]["test_suffix"]
    llm_output_normalized_levenshtein_distance_max = results[
        (dataset_name, model, completion_mode)
    ][f"ID_{id_nums[id_normalized_levenshtein_distance_max]}"]

    matching_length_ratio_mean = np.mean(matching_length_ratio_arr)
    matching_length_ratio_std = np.std(matching_length_ratio_arr)
    matching_length_ratio_max = np.max(matching_length_ratio_arr)
    matching_length_ratio_min = np.min(matching_length_ratio_arr)

    normalized_levenshtein_distance_mean = np.mean(normalized_levenshtein_distance_arr)
    normalized_levenshtein_distance_std = np.std(normalized_levenshtein_distance_arr)
    normalized_levenshtein_distance_max = np.max(normalized_levenshtein_distance_arr)
    normalized_levenshtein_distance_min = np.min(normalized_levenshtein_distance_arr)

    result_data[(dataset_name, model, model_w_params, completion_mode)] = {
        "matching_length_ratio": matching_length_ratio_arr,
        "normalized_levenshtein_distance": normalized_levenshtein_distance_arr,
        "test_prefix_matching_length_ratio_min": test_prefix_matching_length_ratio_min,
        "test_suffix_matching_length_ratio_min": test_suffix_matching_length_ratio_min,
        "llm_output_matching_length_ratio_min": llm_output_matching_length_ratio_min,
        "test_prefix_matching_length_ratio_max": test_prefix_matching_length_ratio_max,
        "test_suffix_matching_length_ratio_max": test_suffix_matching_length_ratio_max,
        "llm_output_matching_length_ratio_max": llm_output_matching_length_ratio_max,
        "test_prefix_normalized_levenshtein_distance_min": test_prefix_normalized_levenshtein_distance_min,
        "test_suffix_normalized_levenshtein_distance_min": test_suffix_normalized_levenshtein_distance_min,
        "llm_output_normalized_levenshtein_distance_min": llm_output_normalized_levenshtein_distance_min,
        "test_prefix_normalized_levenshtein_distance_max": test_prefix_normalized_levenshtein_distance_max,
        "test_suffix_normalized_levenshtein_distance_max": test_suffix_normalized_levenshtein_distance_max,
        "llm_output_normalized_levenshtein_distance_max": llm_output_normalized_levenshtein_distance_max,
    }
    result_scores.append(
        {
            "dataset_name": dataset_name,
            "model": model,
            "model_w_params": model_w_params,
            "completion_mode": completion_mode,
            "time_tag": time_tag,
            "matching_length_ratio_mean": matching_length_ratio_mean,
            "matching_length_ratio_std": matching_length_ratio_std,
            "matching_length_ratio_max": matching_length_ratio_max,
            "matching_length_ratio_min": matching_length_ratio_min,
            "normalized_levenshtein_distance_mean": normalized_levenshtein_distance_mean,
            "normalized_levenshtein_distance_std": normalized_levenshtein_distance_std,
            "normalized_levenshtein_distance_max": normalized_levenshtein_distance_max,
            "normalized_levenshtein_distance_min": normalized_levenshtein_distance_min,
        }
    )

result_scores_df = pd.DataFrame(result_scores)

In [ ]:
# Plot result scores
def plot_result_scores(completion_mode, vars, save_fig=False):
    print(f"completion_mode: {completion_mode}")

    if completion_mode in ["header", "header2"]:
        N = 4
    elif completion_mode in ["row", "row2"]:
        N = 25
    factor = t.ppf(1 - 0.05 / 2, df=N - 1) / np.sqrt(N)

    y_label_dict = {
        "matching_length_ratio_mean": "Mean\nmatching length ratio",
        "matching_length_ratio_max": "Maximum\nmatching length ratio",
        "matching_length_ratio_min": "Minimum\nmatching length ratio",
        "normalized_levenshtein_distance_mean": "Mean normalized\nLevenshtein distance",
        "normalized_levenshtein_distance_max": "Maximum normalized\nLevenshtein distance",
        "normalized_levenshtein_distance_min": "Minimum normalized\nLevenshtein distance",
    }

    ncol = 2
    nsub = len(vars)
    nrow = nsub // ncol + (nsub % ncol > 0)
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 5, nrow * 3), sharex=True)
    axes = axes.ravel()
    handles, labels = None, None
    for i, (var, yrange, ydelta) in enumerate(vars):
        ax = plt.subplot(nrow, ncol, i + 1)

        data = result_scores_df.query("completion_mode == @completion_mode")
        x_labels = data["dataset_name"].unique()
        models = data["model_w_params"].unique()
        x = np.arange(len(x_labels))
        width = 0.8 / len(models)

        for j, model in enumerate(models):
            model_data = data[data["model_w_params"] == model]
            means = model_data[var].to_numpy()
            if "mean" in var:
                cis = model_data[var.replace("_mean", "_std")].to_numpy() * factor
            else:
                cis = None
            ax.bar(
                x + j * width,
                means,
                width=width,
                label=model,
                yerr=cis,
                capsize=1.5 if cis is not None else 0,
                alpha=0.8,
                error_kw={
                    "elinewidth": 1,
                    "capthick": 1,
                },
            )
        ax.grid(axis="y", linestyle="--", alpha=0.3)
        if yrange[0] < 0 < yrange[1]:
            ax.axhline(0, color="black", linewidth=0.5)
        ax.set_ylim(yrange)
        ax.set_yticks(np.arange(yrange[0], yrange[1] + ydelta, ydelta))
        ax.tick_params(axis="y", labelsize=10)
        if var in y_label_dict:
            ylabel = y_label_dict[var]
        else:
            ylabel = var
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_xticks(x + width * (len(models) - 1) / 2)
        ax.set_xticklabels(x_labels, fontsize=10, rotation=90)
        ax.tick_params(axis="x", labelsize=10, rotation=90)
        ax.set_xlabel("", fontsize=12)
        if i == 0:
            handles, labels = ax.get_legend_handles_labels()
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()
    plt.figlegend(
        handles,
        labels,
        loc="upper center",
        fontsize=10,
        bbox_to_anchor=(0.5, 1.08),
        frameon=True,
        ncol=2,
    )
    plt.tight_layout()
    if save_fig:
        plt.savefig(
            f"images/{completion_mode}_metrics.png",
            format="png",
            dpi=300,
            bbox_inches="tight",
        )
        plt.savefig(
            f"images/{completion_mode}_metrics.pdf", format="pdf", bbox_inches="tight"
        )
        plt.savefig(
            f"images/{completion_mode}_metrics.svg", format="svg", bbox_inches="tight"
        )
    plt.show()

In [ ]:
vars = [
    ("matching_length_ratio_mean", (-0.4, 1.6), 0.2),
    ("normalized_levenshtein_distance_mean", (-0.4, 1.6), 0.2),
    ("matching_length_ratio_max", (0.0, 1.0), 0.1),
    ("normalized_levenshtein_distance_max", (0.0, 1.0), 0.1),
    ("matching_length_ratio_min", (0.0, 1.0), 0.1),
    ("normalized_levenshtein_distance_min", (0.0, 1.0), 0.1),
]

plot_result_scores("header", vars, save_fig=True)
plot_result_scores("header2", vars, save_fig=True)

In [ ]:
vars = [
    ("matching_length_ratio_mean", (-0.4, 1.6), 0.2),
    ("normalized_levenshtein_distance_mean", (-0.4, 1.6), 0.2),
    ("matching_length_ratio_max", (0.0, 1.0), 0.1),
    ("normalized_levenshtein_distance_max", (0.0, 1.0), 0.1),
    ("matching_length_ratio_min", (0.0, 1.0), 0.1),
    ("normalized_levenshtein_distance_min", (0.0, 1.0), 0.1),
]

plot_result_scores("row", vars, save_fig=True)
plot_result_scores("row2", vars, save_fig=True)

## Frequency table

In [ ]:
models_w_params_list = result_scores_df["model_w_params"].unique().tolist()
ncol = 3
nsub = len(models_w_params_list)
nrow = (nsub + ncol - 1) // ncol
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 4, nrow * 3))
bins = np.concatenate([np.linspace(0, 1, 11), [1.0 + 1e-8]])
columns = (
    ["dataset_name"]
    + [f"{round(bins[i], 2)}–{round(bins[i + 1], 2)}" for i in range(len(bins) - 2)]
    + ["1.0"]
)
for idx, model_w_params in enumerate(models_w_params_list):
    matching_length_ratio_hist_list = []
    for (dataset_name, _, mwp, cm), data in result_data.items():
        if cm == "row" and mwp == model_w_params:
            matching_length_ratio_arr = data["matching_length_ratio"]
            hist, _ = np.histogram(matching_length_ratio_arr, bins=bins)
            matching_length_ratio_hist_list.append([dataset_name] + hist.tolist())
    df = pd.DataFrame(matching_length_ratio_hist_list, columns=columns)
    if not df.empty:
        df = df.set_index("dataset_name")
    ax = axes.flat[idx]
    sns.heatmap(
        df, cmap="Blues", annot=True, fmt="d", cbar=True, annot_kws={"size": 8}, ax=ax
    )
    ax.collections[0].colorbar.ax.tick_params(labelsize=8)
    ax.set_title(f"{model_w_params}", fontsize=10)
    ax.set_ylabel("", fontsize=10)
    ax.set_xlabel("Matching length ratio bin", fontsize=10)
    ax.set_yticks(np.arange(len(df.index)) + 0.5)
    ax.set_yticklabels(df.index, rotation=0, fontsize=8)
    ax.set_xticks(np.arange(len(df.columns)) + 0.5)
    ax.set_xticklabels(df.columns, rotation=90, fontsize=8)
plt.tight_layout()
plt.savefig(
    "images/matching_length_ratio_frequency_horizontal.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    "images/matching_length_ratio_frequency_horizontal.pdf",
    format="pdf",
    bbox_inches="tight",
)
plt.savefig(
    "images/matching_length_ratio_frequency_horizontal.svg",
    format="svg",
    bbox_inches="tight",
)
plt.show()


In [ ]:
models_w_params_list = [
    result_scores_df["model_w_params"].unique().tolist()[i] for i in [0, 3, 1, 4, 2, 5]
]
ncol = 2
nsub = len(models_w_params_list)
nrow = (nsub + ncol - 1) // ncol
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 4, nrow * 3))
bins = np.concatenate([np.linspace(0, 1, 11), [1.0 + 1e-8]])
columns = (
    ["dataset_name"]
    + [f"{round(bins[i], 2)}–{round(bins[i + 1], 2)}" for i in range(len(bins) - 2)]
    + ["1.0"]
)
for idx, model_w_params in enumerate(models_w_params_list):
    matching_length_ratio_hist_list = []
    for (dataset_name, _, mwp, cm), data in result_data.items():
        if cm == "row" and mwp == model_w_params:
            matching_length_ratio_arr = data["matching_length_ratio"]
            hist, _ = np.histogram(matching_length_ratio_arr, bins=bins)
            matching_length_ratio_hist_list.append([dataset_name] + hist.tolist())
    df = pd.DataFrame(matching_length_ratio_hist_list, columns=columns)
    if not df.empty:
        df = df.set_index("dataset_name")
    ax = axes.flat[idx]
    sns.heatmap(
        df, cmap="Blues", annot=True, fmt="d", cbar=True, annot_kws={"size": 8}, ax=ax
    )
    ax.collections[0].colorbar.ax.tick_params(labelsize=8)
    ax.set_title(f"{model_w_params}", fontsize=10)
    ax.set_ylabel("", fontsize=10)
    ax.set_xlabel("Matching length ratio bin", fontsize=10)
    ax.set_yticks(np.arange(len(df.index)) + 0.5)
    ax.set_yticklabels(df.index, rotation=0, fontsize=8)
    ax.set_xticks(np.arange(len(df.columns)) + 0.5)
    ax.set_xticklabels(df.columns, rotation=90, fontsize=8)
plt.tight_layout()
plt.savefig(
    "images/matching_length_ratio_frequency_vertical.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    "images/matching_length_ratio_frequency_vertical.pdf",
    format="pdf",
    bbox_inches="tight",
)
plt.savefig(
    "images/matching_length_ratio_frequency_vertical.svg",
    format="svg",
    bbox_inches="tight",
)
plt.show()